# Objective 2 (Experiment E3) — Authoritative Controlled Synthetic Drift-Regime Execution Engine

**Thesis:** *Adaptive Low-Latency Fraud Detection in Streaming Financial Systems with LLM-Augmented Explainability*  
**Experiment:** Objective 2: Drift-Regime Interaction (Experiment E3: RQ2 / Hypothesis H2)  
**Methodology:** Controlled Synthetic Distribution Shift, Prequential Evaluation (Test-Then-Train), Discrete Event Diagnostics, Atomic Checkpointing & Resumable Manifest  

### Research Question & Hypothesis
- **RQ2:** *How does the effectiveness of adaptation policies ($P_0–P_3$) vary across distinct concept-drift regimes, including sudden, gradual, recurring, and localized drift?*
- **Hypothesis H2:** *The relative predictive rank and operational efficiency of adaptation policies differs significantly across drift regimes (Policy $\times$ Drift Regime interaction effect).*

### Authoritative Scientific Framing (Post-Validation)
Experiment E3 implements a **controlled synthetic distribution shift with a measurable conditional relationship shift $P(Y \mid X)$, accompanied by covariate shift for the affected feature variables** ($P_A(X,Y) \to P_B(X,Y)$).  
Empirically validated properties (`docs/internal/objective2_e3_injection_validation.md`):
1. **$P(Y \mid X)$ Degradation:** Fixed Concept A Hoeffding Tree experiences an immediate $-34.19\%$ relative drop in Average Precision ($0.13138 \to 0.08646$).
2. **Absolute Label Integrity:** Target labels $Y$ are 100% byte-for-byte identical ($\Delta\text{prevalence} = 0.0000000000$).
3. **Feature Coverage:** 100% of rows modified in Concept B.
4. **Seed Stochasticity Distinction:**
   - **Gradual Drift:** 10 seeds produce 10 unique transition streams (genuine stochastic replicate realizations).
   - **Sudden / Recurring / Localized Drift:** 10 nominal seeds produce 1 unique transition sequence (deterministic reproducibility and numerical invariance checks, $s_D = 0$). Downstream inferential statistics must **NOT** treat deterministic seeds as independent degrees of freedom.

## 1. Import Required Libraries and Environment Setup

Bootstrap streaming dependencies (`river`, `narwhals`) and initialize the environment.

In [1]:
import os
import sys
import time
import json
import hashlib
import subprocess
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Automatic streaming dependency bootstrap for cloud/Kaggle environments
try:
    import river
except ImportError:
    print("[ENVIRONMENT] river not found. Installing river and narwhals via pip (--no-deps)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "river", "--no-deps", "narwhals", "-q"
    ])
    import river

import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python executable: {sys.executable}")
print(f"NumPy version:     {np.__version__}")
print(f"Pandas version:    {pd.__version__}")
print(f"River version:     {river.__version__}")
print(f"Scikit-learn:      {sklearn.__version__}")

Python executable: C:\Projects\Thesis\.venv\Scripts\python.exe
NumPy version:     2.5.1
Pandas version:    3.0.3
River version:     0.26.1
Scikit-learn:      1.9.0


## 2. Execution Mode Selection & Job Configuration

Choose between `SMOKE` (fast local verification subset) and `FULL` (authoritative 160-job execution matrix).

In [2]:
# TOP-LEVEL EXECUTION MODE SELECTION
# Options:
#   "SMOKE" : Fast local verification mode (1,000 rows, exercises all 4 policies & 4 regimes)
#   "FULL"  : Authoritative Kaggle/Production execution (160 runs over full stream)
RUN_MODE = "SMOKE"

# SHARED HYPERPARAMETERS (Locked by DEC-01, DEC-02, DEC-04, E3 Spec)
GRACE_PERIOD = 50 if RUN_MODE == "SMOKE" else 200
DELTA = 0.002
N_INTERVAL = 1000 if RUN_MODE == "SMOKE" else 10000
WARMUP_FRAC = 0.15
WINDOW_SIZE = 10000
DIAGNOSTIC_HORIZON = 500

if RUN_MODE == "SMOKE":
    SMOKE_ROW_LIMIT = 1000   # Fast local verification slice
    ROLLING_WINDOW_SIZE = 200
    MAX_TRAJECTORY_POINTS = 50
    E3_T_DRIFT = 300
    E3_TRANSITION_WIDTH = 200
    E3_RECURRENCE_PERIOD = 200
    E3_HORIZON = 200
    
    # Smoke matrix: 5 jobs exercising all 4 policies, all 4 regimes, and seed variation
    JOB_SPECS = [
        {"policy": "P0", "regime": "sudden",     "window_size": WINDOW_SIZE, "seed": 42},
        {"policy": "P1", "regime": "gradual",    "window_size": WINDOW_SIZE, "seed": 42},
        {"policy": "P2", "regime": "recurring",  "window_size": WINDOW_SIZE, "seed": 42},
        {"policy": "P3", "regime": "localized",  "window_size": WINDOW_SIZE, "seed": 42},
        {"policy": "P1", "regime": "gradual",    "window_size": WINDOW_SIZE, "seed": 101},
    ]
else:
    SMOKE_ROW_LIMIT = None   # Full transaction stream
    ROLLING_WINDOW_SIZE = 5000
    MAX_TRAJECTORY_POINTS = 200
    E3_T_DRIFT = 25000
    E3_TRANSITION_WIDTH = 5000
    E3_RECURRENCE_PERIOD = 25000
    E3_HORIZON = 5000
    
    # AUTHORITATIVE 160-JOB E3 MATRIX
    # 4 Regimes x 4 Policies x 10 Seeds = 160 jobs
    # Deterministic Execution Order: regime -> policy -> seed
    REGIMES = ["sudden", "gradual", "recurring", "localized"]
    POLICIES = ["P0", "P1", "P2", "P3"]
    SEEDS = [42, 101, 123, 256, 314, 420, 555, 666, 777, 909]
    
    JOB_SPECS = []
    for reg in REGIMES:
        for pol in POLICIES:
            for s in SEEDS:
                JOB_SPECS.append({
                    "policy": pol,
                    "regime": reg,
                    "window_size": WINDOW_SIZE,
                    "seed": s,
                })

print(f"Configured RUN_MODE:      {RUN_MODE}")
print(f"Total Target Jobs:        {len(JOB_SPECS)}")
if RUN_MODE == "SMOKE":
    for spec in JOB_SPECS:
        print(f"  - {spec['policy']} {spec['regime']} W{spec['window_size']} seed={spec['seed']}")
else:
    print("  [FULL MODE CONFIGURED: 160 jobs ready for Kaggle/cloud execution]")

Configured RUN_MODE:      SMOKE
Total Target Jobs:        5
  - P0 sudden W10000 seed=42
  - P1 gradual W10000 seed=42
  - P2 recurring W10000 seed=42
  - P3 localized W10000 seed=42
  - P1 gradual W10000 seed=101


## 3. Project Root & Directory Hierarchy Configuration

Resolve workspace paths and configure dedicated Objective 2 / E3 output directories.

In [3]:
def find_project_root(start_path: Path) -> Optional[Path]:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "src").is_dir() and (current / "pyproject.toml").is_file():
            return current
        current = current.parent
    return None

PROJECT_ROOT = find_project_root(Path.cwd())

# On Kaggle, clone repo into /kaggle/working if src is not present in cwd
kaggle_working = Path("/kaggle/working")
if kaggle_working.exists() and (PROJECT_ROOT is None or not (PROJECT_ROOT / "src").is_dir()):
    REPO_NAME = "adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability"
    REPO_DIR = kaggle_working / REPO_NAME
    REPO_URL = "https://github.com/ParminderSinghGithub/adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability.git"
    if not (REPO_DIR / "src").is_dir():
        print(f"[KAGGLE] Cloning repository into {REPO_DIR}...")
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        print(f"[KAGGLE] Pulling latest repository updates in {REPO_DIR}...")
        os.system(f"cd {REPO_DIR} && git pull origin main")
    PROJECT_ROOT = REPO_DIR

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(".").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root resolved: {PROJECT_ROOT}")

# Dedicated E3 Outputs Hierarchy (isolated from Objective 1 namespaces)
OUTPUTS_BASE = kaggle_working if kaggle_working.exists() else PROJECT_ROOT
CHECKPOINT_DIR = OUTPUTS_BASE / "outputs" / "checkpoints" / "objective2_e3_runs"
MANIFEST_PATH = OUTPUTS_BASE / "outputs" / "checkpoints" / "objective2_e3_manifest.json"
ZIP_ARCHIVE_PATH = OUTPUTS_BASE / "outputs" / "objective2_e3_artifacts_live.zip"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint directory:   {CHECKPOINT_DIR}")
print(f"Manifest path:          {MANIFEST_PATH}")
print(f"Live archive mirror:    {ZIP_ARCHIVE_PATH}")

Project root resolved: C:\Projects\Thesis
Checkpoint directory:   C:\Projects\Thesis\outputs\checkpoints\objective2_e3_runs
Manifest path:          C:\Projects\Thesis\outputs\checkpoints\objective2_e3_manifest.json
Live archive mirror:    C:\Projects\Thesis\outputs\objective2_e3_artifacts_live.zip


## 4. Benchmark Dataset Discovery & Ingestion

Locate the authoritative IEEE-CIS benchmark dataset in `/kaggle/input` or local workspace.

In [4]:
def resolve_ieee_cis_dir() -> Path:
    # 1. Search recursively across ALL attached datasets in /kaggle/input
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob("train_transaction.csv"))
        if matches:
            return matches[0].parent

    # 2. Search workspace directories (local or /kaggle/working)
    for base in [PROJECT_ROOT, Path.cwd(), Path("/kaggle/working")]:
        if base.exists():
            matches = list(base.rglob("train_transaction.csv"))
            if matches:
                return matches[0].parent

    available = list(kaggle_input.glob("*")) if kaggle_input.exists() else []
    raise FileNotFoundError(
        f"Could not locate 'train_transaction.csv'. "
        f"Attached items in /kaggle/input: {[p.name for p in available]}"
    )

IEEE_CIS_DIR = resolve_ieee_cis_dir()
print(f"IEEE-CIS data source resolved at: {IEEE_CIS_DIR}")

from src.data.loader import load_ieee_cis
from src.utils.seed import set_seed

df_raw = load_ieee_cis(data_dir=IEEE_CIS_DIR, split="train", join_identity=True)
if SMOKE_ROW_LIMIT is not None:
    df_raw = df_raw.iloc[:SMOKE_ROW_LIMIT].copy()
print(f"Ingested raw transactions: {len(df_raw):,} rows, {len(df_raw.columns)} columns")

IEEE-CIS data source resolved at: C:\Projects\Thesis\datasets\ieee_cis


Ingested raw transactions: 1,000 rows, 434 columns


## 5. Chronological Verification & Warmup-Only Preprocessing

Verify timestamp monotonicity and fit streaming feature preprocessor on warmup partition only (preventing future leakage).

In [5]:
# Verify strict chronological monotonicity of TransactionDT
dt_series = df_raw["TransactionDT"]
is_monotonic = bool(dt_series.is_monotonic_increasing)
print(f"Chronological Monotonicity Check (TransactionDT): {is_monotonic}")
assert is_monotonic, "Dataset MUST be sorted monotonically by TransactionDT!"

# Compute dataset fingerprint for auditability
data_fingerprint = hashlib.sha256(
    f"{len(df_raw)}_{df_raw['TransactionID'].iloc[0]}_{df_raw['TransactionID'].iloc[-1]}".encode("utf-8")
).hexdigest()[:16]
print(f"Dataset fingerprint: {data_fingerprint}")

# Compute warmup split
n_total = len(df_raw)
warmup_size = int(n_total * WARMUP_FRAC)
stream_size = n_total - warmup_size
print(f"Warmup size (Phase 1): {warmup_size:,} ({WARMUP_FRAC*100:.1f}%)")
print(f"Stream size (Phase 2): {stream_size:,} ({(1-WARMUP_FRAC)*100:.1f}%)")

# Preprocessing: Fit ONLY on warmup partition
from src.data.preprocessing import StreamingPreprocessor

preprocessor = StreamingPreprocessor(
    dataset_name="ieee_cis",
    scale_features=False,
)

print("Fitting streaming preprocessor on warmup partition only...")
preprocessor.fit(df_raw.iloc[:warmup_size])
print(f"Preprocessor fitted. Total features after transform: {len(preprocessor.feature_names):,}")

print("Transforming full dataset...")
X_all, y_all, seg_all = preprocessor.transform(df_raw)

print(f"Feature matrix X: {X_all.shape}")
print(f"Target Series y:  {len(y_all):,} (fraud rate = {y_all.mean():.4f})")
print(f"Segment Series:   {seg_all.nunique() if seg_all is not None else 0} distinct segments")

Chronological Monotonicity Check (TransactionDT): True
Dataset fingerprint: af2d52cfe83db380
Warmup size (Phase 1): 150 (15.0%)
Stream size (Phase 2): 850 (85.0%)
Fitting streaming preprocessor on warmup partition only...


Preprocessor fitted. Total features after transform: 431
Transforming full dataset...


Feature matrix X: (1000, 431)
Target Series y:  1,000 (fraud rate = 0.0150)
Segment Series:   5 distinct segments


## 6. Manifest Initialization & Resumption Discovery

Initialize the atomic `E3ExperimentManifest` manager to track job lifecycle and skip already completed runs.

In [6]:
from src.pipeline.manifest import E3ExperimentManifest

manifest = E3ExperimentManifest(
    manifest_path=MANIFEST_PATH,
    artifacts_dir=CHECKPOINT_DIR,
    zip_archive_path=ZIP_ARCHIVE_PATH,
    experiment_name="Objective2_E3",
    dataset_name="IEEE-CIS",
)

print(f"Loaded E3 manifest with {len(manifest.jobs)} known jobs.")
completed_count = sum(1 for j in manifest.jobs.values() if j.status == "COMPLETED")
print(f"Completed jobs on disk: {completed_count}")

Loaded E3 manifest with 0 known jobs.
Completed jobs on disk: 0


## 7. Trajectory Downsampling Helper

Compact rolling trajectory extractor for visualization in analysis notebooks.

In [7]:
from sklearn.metrics import precision_recall_curve, auc

def extract_downsampled_trajectories(records: List[Any], window_size: int = 5000, max_points: int = 150) -> Dict[str, Any]:
    """Extract rolling PR-AUC trajectory downsampled to a compact representation for serialization."""
    n_records = len(records)
    if n_records == 0:
        return {"indices": [], "rolling_pr_auc": []}

    step = max(1, n_records // max_points)
    indices = []
    pr_aucs = []

    y_true_all = [r.y_true for r in records]
    y_prob_all = [r.y_prob for r in records]

    for end_idx in range(window_size, n_records + 1, step):
        start_idx = end_idx - window_size
        y_w = y_true_all[start_idx:end_idx]
        p_w = y_prob_all[start_idx:end_idx]

        if sum(y_w) > 0 and len(y_w) - sum(y_w) > 0:
            prec, rec, _ = precision_recall_curve(y_w, p_w)
            score = float(auc(rec, prec))
        else:
            score = 0.0

        indices.append(end_idx)
        pr_aucs.append(round(score, 5))

    return {
        "indices": indices,
        "rolling_pr_auc": pr_aucs,
        "window_size": window_size,
    }

print("Trajectory downsampling helper defined.")

Trajectory downsampling helper defined.


## 8. Authoritative E3 Job Execution Loop (Resumable, Atomic, & Gated)

Orchestrates drift injection (`src.drift.injection`), prequential streaming (`src.pipeline.runner`), discrete event-centered evaluation (`src.evaluation.event_metrics`), and atomic artifact persistence.

In [8]:
from src.drift.injection import DriftRegime, E3DriftConfig, SyntheticDriftInjector
from src.evaluation.event_metrics import compute_event_centered_metrics
from src.pipeline.runner import PrequentialRunner

total_jobs = len(JOB_SPECS)
job_idx = 0
skipped_count = 0
completed_in_run = 0
failed_count = 0

print(f"Starting E3 execution loop over {total_jobs} target jobs (RUN_MODE={RUN_MODE})...")

for spec in JOB_SPECS:
    job_idx += 1
    policy = spec["policy"]
    regime = spec["regime"]
    seed = spec["seed"]
    window_size = spec["window_size"]
    job_id = manifest.make_job_id(policy, regime, seed, window_size=window_size)

    # 1. Resumption Guard: Skip if already completed
    if manifest.is_job_completed(policy, regime, seed, window_size=window_size):
        print(f"[{job_idx}/{total_jobs}] Job {job_id} already COMPLETED on disk -> SKIPPING.")
        skipped_count += 1
        continue

    print(f"[{job_idx}/{total_jobs}] Executing Job {job_id}...")
    manifest.record_start(policy, regime, seed, window_size=window_size, dataset_fingerprint=data_fingerprint)
    set_seed(seed)

    # 2. Configure E3 Synthetic Drift Injector
    drift_cfg = E3DriftConfig(
        regime=regime,
        t_drift=E3_T_DRIFT,
        transition_width=E3_TRANSITION_WIDTH,
        recurrence_period=E3_RECURRENCE_PERIOD,
        target_segment="C",
        segment_col="ProductCD",
        feature_pairs=[("C1", "C2"), ("C3", "C4"), ("card1", "card2")],
        window_size=window_size,
        seed=seed,
    )
    injector = SyntheticDriftInjector(drift_cfg)

    # 3. Inject synthetic distribution shift on stream (warmup partition untouched)
    X_drift, y_drift, seg_drift, metadata_list = injector.inject_dataframe(
        X=X_all,
        y=y_all,
        segment=seg_all,
        warmup_size=warmup_size,
    )

    config_snapshot = {
        "warmup_size": warmup_size,
        "stream_size": stream_size,
        "warmup_frac": WARMUP_FRAC,
        "window_size": window_size,
        "n_interval": N_INTERVAL,
        "grace_period": GRACE_PERIOD,
        "delta": DELTA,
        "diagnostic_horizon": DIAGNOSTIC_HORIZON,
        "t_drift": E3_T_DRIFT,
        "horizon": E3_HORIZON,
        "regime": regime,
        "run_mode": RUN_MODE,
    }

    t_start = time.perf_counter()
    try:
        # 4. Instantiate Prequential Runner
        runner = PrequentialRunner.from_config(
            policy_str=policy,
            detector_str="adwin",
            grace_period=GRACE_PERIOD,
            delta=DELTA,
            window_size=window_size,
            n_interval=N_INTERVAL,
            detector_kwargs={"delta": DELTA},
            segment_aware=(policy == "P3"),
            seed=seed,
            p0_mode="incremental",
            diagnostic_horizon=DIAGNOSTIC_HORIZON,
        )

        # 5. Execute prequential evaluation over drifting stream
        run_result = runner.run(
            X=X_drift,
            y=y_drift,
            segment=seg_drift,
            warmup_size=warmup_size,
        )

        wall_clock_s = time.perf_counter() - t_start

        # 6. Compute discrete event-centered metrics
        event_metrics = compute_event_centered_metrics(
            records=run_result.records,
            drift_event_log=run_result.drift_event_log,
            adaptation_log=run_result.adaptation_log,
            drift_config=drift_cfg,
            horizon=E3_HORIZON,
            recovery_window=min(1000, max(50, E3_HORIZON // 2)) if RUN_MODE == "SMOKE" else 1000,
            recovery_tolerance=0.05,
            filter_segment="C" if regime == "localized" else None,
        )

        # 7. Downsample trajectories for compact storage
        trajectories = extract_downsampled_trajectories(
            run_result.records,
            window_size=ROLLING_WINDOW_SIZE,
            max_points=MAX_TRAJECTORY_POINTS,
        )

        # 8. Persist artifact atomically and update manifest
        artifact_path = manifest.record_completion(
            policy=policy,
            regime=regime,
            seed=seed,
            run_result=run_result,
            event_metrics=event_metrics,
            wall_clock_duration_s=wall_clock_s,
            drift_config=drift_cfg,
            dataset_fingerprint=data_fingerprint,
            code_version="authoritative_e3_v1",
            config_snapshot=config_snapshot,
            trajectories=trajectories,
            window_size=window_size,
        )

        completed_in_run += 1
        fm = run_result.final_metrics
        lp = run_result.latency_percentiles
        pr_val = fm.pr_auc if hasattr(fm, "pr_auc") else fm.get("pr_auc", 0.0)
        p95_val = lp.get("p95_ms", lp.get("p95", 0.0) * 1000)
        em_dict = event_metrics.as_dict()
        pre_pr = em_dict.get("pre_drift_pr_auc")
        post_pr = em_dict.get("post_drift_pr_auc")
        d_pr = em_dict.get("delta_pr_auc")
        d_delay = em_dict.get("detection_delay")

        pre_str = f"{pre_pr:.4f}" if pre_pr is not None else "N/A"
        post_str = f"{post_pr:.4f}" if post_pr is not None else "N/A"
        d_str = f"{d_pr:+.4f}" if d_pr is not None else "N/A"
        delay_str = str(d_delay) if d_delay is not None else "none"

        print(
            f"    [COMPLETED] Duration: {wall_clock_s:.1f}s | "
            f"PR-AUC: {pr_val:.4f} | Pre: {pre_str} | Post: {post_str} | Delta: {d_str} | "
            f"Delay: {delay_str} | Adaptations: {len(run_result.adaptation_log)} | "
            f"Latency p95: {p95_val:.2f}ms | Saved: {artifact_path.name}"
        )

    except Exception as ex:
        failed_count += 1
        manifest.record_failure(policy, regime, seed, str(ex), window_size=window_size)
        print(f"    [FAILED] Job {job_id} encountered error: {ex}")
        raise

print("\nExecution loop complete.")
print(f"Total: {total_jobs} | Completed this run: {completed_in_run} | Skipped/Resumed: {skipped_count} | Failed: {failed_count}")

Starting E3 execution loop over 5 target jobs (RUN_MODE=SMOKE)...
[1/5] Executing Job e3_P0_sudden_W10000_seed42...


    [COMPLETED] Duration: 2.0s | PR-AUC: 0.0127 | Pre: 0.0360 | Post: 0.0085 | Delta: -0.0275 | Delay: none | Adaptations: 0 | Latency p95: 0.04ms | Saved: run_e3_P0_sudden_W10000_seed42.json
[2/5] Executing Job e3_P1_gradual_W10000_seed42...


    [COMPLETED] Duration: 1.7s | PR-AUC: 0.0127 | Pre: 0.0360 | Post: 0.0085 | Delta: -0.0275 | Delay: none | Adaptations: 0 | Latency p95: 0.02ms | Saved: run_e3_P1_gradual_W10000_seed42.json
[3/5] Executing Job e3_P2_recurring_W10000_seed42...


    [COMPLETED] Duration: 2.1s | PR-AUC: 0.0127 | Pre: 0.0360 | Post: 0.0085 | Delta: -0.0275 | Delay: none | Adaptations: 0 | Latency p95: 0.05ms | Saved: run_e3_P2_recurring_W10000_seed42.json
[4/5] Executing Job e3_P3_localized_W10000_seed42...


    [COMPLETED] Duration: 1.9s | PR-AUC: 0.1075 | Pre: 0.2665 | Post: 0.0417 | Delta: -0.2248 | Delay: none | Adaptations: 0 | Latency p95: 0.03ms | Saved: run_e3_P3_localized_W10000_seed42.json
[5/5] Executing Job e3_P1_gradual_W10000_seed101...


    [COMPLETED] Duration: 1.6s | PR-AUC: 0.0127 | Pre: 0.0360 | Post: 0.0085 | Delta: -0.0275 | Delay: none | Adaptations: 0 | Latency p95: 0.02ms | Saved: run_e3_P1_gradual_W10000_seed101.json

Execution loop complete.
Total: 5 | Completed this run: 5 | Skipped/Resumed: 0 | Failed: 0


## 9. Execution Summary Table & Checkpoint Verification

Display summary table of all recorded E3 jobs and audit completion integrity.

In [9]:
summary_df = manifest.get_summary_dataframe()
display_cols = [
    "job_id", "policy", "regime", "seed", "status",
    "pr_auc", "pre_drift_pr_auc", "post_drift_pr_auc", "delta_pr_auc",
    "detection_delay", "adaptation_count", "latency_p95_ms", "wall_clock_duration_s"
]
available_cols = [c for c in display_cols if c in summary_df.columns]
print(f"\nExecution Manifest Summary ({len(summary_df)} total jobs recorded):")
if not summary_df.empty:
    print(summary_df[available_cols].to_string(index=False))
else:
    print("No jobs recorded in manifest.")

# Validation check
if RUN_MODE == "FULL":
    completed = sum(1 for j in manifest.jobs.values() if j.status == "COMPLETED")
    assert completed == 160, f"Expected 160 completed jobs in FULL mode, found {completed}."
    print("ALL 160 FULL-MODE JOBS SUCCESSFULLY COMPLETED & VERIFIED.")
else:
    print(f"\nSMOKE MODE VALIDATION COMPLETE: {len(summary_df)} jobs verified successfully.")


Execution Manifest Summary (5 total jobs recorded):
                       job_id policy    regime  seed    status   pr_auc  pre_drift_pr_auc  post_drift_pr_auc  delta_pr_auc detection_delay  adaptation_count  latency_p95_ms  wall_clock_duration_s
   e3_P0_sudden_W10000_seed42     P0    sudden    42 COMPLETED 0.012738           0.03602            0.00850      -0.02752            None                 0          0.0412                  1.992
  e3_P1_gradual_W10000_seed42     P1   gradual    42 COMPLETED 0.012738           0.03602            0.00850      -0.02752            None                 0          0.0168                  1.739
e3_P2_recurring_W10000_seed42     P2 recurring    42 COMPLETED 0.012738           0.03602            0.00850      -0.02752            None                 0          0.0466                  2.137
e3_P3_localized_W10000_seed42     P3 localized    42 COMPLETED 0.107457           0.26652            0.04167      -0.22485            None                 0       